<a href="https://colab.research.google.com/github/ArifahZhafirah/PEMBELAJARAN-MESIN/blob/main/JS05/TugasLab-02/PEMMES%20TUGAS%20LAB%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langkah 1 - Load data

In [12]:
import numpy as np
import pandas as pd

df = pd.read_csv('spam.csv', encoding='latin-1')

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


#Langkah 2 - Preprocessing: drop kolom dan ubah nama kolom

In [13]:
# Drop 3 kolom terakhir dengan fungsi iloc
df = df.drop(df.iloc[:,2:], axis=1)

# Ubah nama kolom v1 (label) dan v2 (teks sms)
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})

# Cek data
df.head()

,Labels,SMS
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


#Langkah 3 - Inspeksi data

In [14]:
# Cek Jumlah Data Per Kelas
print(df['Labels'].value_counts())
print('\n')

# Cek Kelengkapan Data
print(df.info())
print('\n')

# Cek Statistik Deskriptif
print(df.describe())

Labels
ham     4825
spam     747
Name: count, dtype: int64


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Labels  5572 non-null   object
 1   SMS     5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


       Labels                     SMS
count    5572                    5572
unique      2                    5169
top       ham  Sorry, I'll call later
freq     4825                      30


#Langah 4 - Encoding label

In [15]:
# Data untuk label
new_labels = {
    'spam': 1,
    'ham': 0
}

# Encode label
df['Labels'] = df['Labels'].map(new_labels)

# Cek data
df.head()

,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


#Langkah 5 - Pisahkan fitur dengan label

In [16]:
X = df['SMS'].values
y = df['Labels'].values

#Soal 1

#Langkah 1 - ekstraksi fitur CountVectorizer dengan stop_words

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Split data training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

# Inisiasi CountVectorizer dengan stop_words diaktifkan
bow = CountVectorizer(stop_words='english')

# Fitting dan transform X_train dengan CountVectorizer
X_train = bow.fit_transform(X_train)

# Transform X_test
X_test = bow.transform(X_test)

# Cek fitur
print(len(bow.get_feature_names_out()))
print(f'Dimensi data: {X_train.shape}')

7466
Dimensi data: (4457, 7466)


#Langkah 2 - training dan evaluasi Multinomial Naive Bayes

In [18]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Inisiasi MultinomialNB
mnb = MultinomialNB()

# Fit model
mnb.fit(X_train, y_train)

# Prediksi dengan data training
y_pred_train = mnb.predict(X_train)

# Evaluasi akurasi data training
acc_train = accuracy_score(y_train, y_pred_train)

# Prediksi dengan data testing
y_pred_test = mnb.predict(X_test)

# Evaluasi akurasi data testing
acc_test = accuracy_score(y_test, y_pred_test)

# Print hasil evaluasi
print(f'Hasil akurasi data train: {acc_train}')
print(f'Hasil akurasi data test: {acc_test}')
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_test))
print("Laporan Klasifikasi:\n", classification_report(y_test, y_pred_test))

Hasil akurasi data train: 0.9946152120260264
Hasil akurasi data test: 0.9829596412556054
Confusion Matrix:
 [[951   3]
 [ 16 145]]
Laporan Klasifikasi:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       954
           1       0.98      0.90      0.94       161

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



Model Multinomial Naive Bayes dengan fitur CountVectorizer mendapat akurasi 0.9946 pada data train dan 0.9830 pada data test. Selisihnya kecil, jadi model tidak overfitting. Dari 954 SMS ham, 951 diprediksi benar dan hanya 3 yang salah dianggap spam. Dari 161 SMS spam, 145 terdeteksi dan 16 lolos sebagai ham. Nilai recall spam 0.90 dan f1-score spam 0.94.

#Soal 2

#Langkah 1 - ekstraksi fitur TF-IDF dengan stop_words

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Split ulang data teks dengan random_state yang sama seperti sebelumnya
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

# Inisiasi TfidfVectorizer dengan stop_words diaktifkan
tfidf = TfidfVectorizer(stop_words='english')

# Fitting dan transform X_train dengan TfidfVectorizer
X_train = tfidf.fit_transform(X_train)

# Transform X_test
X_test = tfidf.transform(X_test)

# Cek fitur
print(len(tfidf.get_feature_names_out()))
print(f'Dimensi data: {X_train.shape}')

7466
Dimensi data: (4457, 7466)


#Langkah 2 - training dan evaluasi Multinomial Naive Bayes (TF-IDF)

In [20]:
# Inisiasi MultinomialNB
mnb_tfidf = MultinomialNB()

# Fit model
mnb_tfidf.fit(X_train, y_train)

# Prediksi dengan data training
y_pred_train_tfidf = mnb_tfidf.predict(X_train)

# Evaluasi akurasi data training
acc_train_tfidf = accuracy_score(y_train, y_pred_train_tfidf)

# Prediksi dengan data testing
y_pred_test_tfidf = mnb_tfidf.predict(X_test)

# Evaluasi akurasi data testing
acc_test_tfidf = accuracy_score(y_test, y_pred_test_tfidf)

# Print hasil evaluasi
print(f'Hasil akurasi data train (TF-IDF): {acc_train_tfidf}')
print(f'Hasil akurasi data test (TF-IDF): {acc_test_tfidf}')
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_test_tfidf))
print("Laporan Klasifikasi:\n", classification_report(y_test, y_pred_test_tfidf))

Hasil akurasi data train (TF-IDF): 0.9842943684092439
Hasil akurasi data test (TF-IDF): 0.9605381165919282
Confusion Matrix:
 [[954   0]
 [ 44 117]]
Laporan Klasifikasi:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       954
           1       1.00      0.73      0.84       161

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



#Sel 3 - Perbandingan CountVectorizer dan TF-IDF

In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Membuat tabel perbandingan kedua fitur
hasil = pd.DataFrame({
    'Fitur': ['CountVectorizer', 'TF-IDF'],
    'Akurasi Train': [acc_train, acc_train_tfidf],
    'Akurasi Test': [acc_test, acc_test_tfidf],
    'Precision Spam': [precision_score(y_test, y_pred_test), precision_score(y_test, y_pred_test_tfidf)],
    'Recall Spam': [recall_score(y_test, y_pred_test), recall_score(y_test, y_pred_test_tfidf)],
    'F1-Score Spam': [f1_score(y_test, y_pred_test), f1_score(y_test, y_pred_test_tfidf)]
})

# Membulatkan 4 angka di belakang koma
hasil = hasil.round(4)
hasil

,Fitur,Akurasi Train,Akurasi Test,Precision Spam,Recall Spam,F1-Score Spam
0,CountVectorizer,0.9946,0.9830,0.9797,0.9006,0.9385
1,TF-IDF,0.9843,0.9605,1.0000,0.7267,0.8417


Dengan fitur TF-IDF, akurasi train 0.9843 dan akurasi test 0.9605. Tidak ada SMS ham yang salah dianggap spam (precision spam 1.00), tetapi hanya 117 dari 161 SMS spam yang terdeteksi. Sebanyak 44 SMS spam lolos sebagai ham, sehingga recall spam turun menjadi 0.73 dan f1-score spam menjadi 0.84.

#Perbandingan dan Kesimpulan

Kedua fitur memakai split data yang sama dan jumlah fitur yang sama (7466 kata), jadi perbandingannya adil. CountVectorizer mendapat akurasi test 0.9830, lebih tinggi dari TF-IDF yang mendapat 0.9605. Selisih akurasinya tidak besar, tetapi pada kelas spam perbedaannya cukup jelas. CountVectorizer berhasil mendeteksi 145 SMS spam (recall 0.90, f1-score 0.94), sedangkan TF-IDF hanya 117 (recall 0.73, f1-score 0.84). TF-IDF memang tidak salah menandai satu pun SMS ham sebagai spam, tetapi lebih banyak spam yang lolos, padahal tujuan utama klasifikasi ini adalah menangkap spam. Data juga tidak seimbang (ham 4825, spam 747), sehingga kelas spam lebih mudah terlewat.

Jadi, untuk data spam.csv dengan pengaturan ini, fitur terbaik adalah CountVectorizer karena akurasinya lebih tinggi dan performanya pada kelas spam lebih baik. Kesimpulan ini berlaku untuk pembagian data random_state=50 dan parameter bawaan, sehingga hasilnya bisa sedikit berbeda kalau pengaturannya diubah.